**Installing the packages**

In [ ]:
!pip install -q segmentation-models-pytorch timm albumentations opencv-python matplotlib pillow

**Cloning the YOLOv5**

In [ ]:
!git clone https://github.com/ultralytics/yolov5
%cd yolov5
!pip install -qr requirements.txt

**Importing the necessary libraries**

In [ ]:
import os
import time
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image
import torch.nn.functional as F
import segmentation_models_pytorch as smp

**Setting the image path**

In [ ]:
IMAGE_PATH = "/content/test_image.jpg"

if not os.path.exists(IMAGE_PATH):
    raise FileNotFoundError(f"Image not found: {IMAGE_PATH}")

print("Using image:", IMAGE_PATH)

**Selecting the device**

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

**Discoverin the Cityscapes classes and colors**

In [ ]:
NUM_CLASSES = 19

CITYSCAPES_CLASSES = [
    'road', 'sidewalk', 'building', 'wall', 'fence', 'pole',
    'traffic light', 'traffic sign', 'vegetation', 'terrain',
    'sky', 'person', 'rider', 'car', 'truck', 'bus',
    'train', 'motorcycle', 'bicycle'
]

CITYSCAPES_COLORS = np.array([
    [128,  64, 128],  # road
    [244,  35, 232],  # sidewalk
    [ 70,  70,  70],  # building
    [102, 102, 156],  # wall
    [190, 153, 153],  # fence
    [153, 153, 153],  # pole
    [250, 170,  30],  # traffic light
    [220, 220,   0],  # traffic sign
    [107, 142,  35],  # vegetation
    [152, 251, 152],  # terrain
    [ 70, 130, 180],  # sky
    [220,  20,  60],  # person
    [255,   0,   0],  # rider
    [  0,   0, 142],  # car
    [  0,   0,  70],  # truck
    [  0,  60, 100],  # bus
    [  0,  80, 100],  # train
    [  0,   0, 230],  # motorcycle
    [119,  11,  32],  # bicycle
], dtype=np.uint8)

**Loading the original image**

In [ ]:
image = Image.open(IMAGE_PATH).convert("RGB")
image_np = np.array(image)
orig_h, orig_w = image_np.shape[:2]

print("Original image shape:", image_np.shape)

plt.figure(figsize=(8,5))
plt.imshow(image_np)
plt.title("Original Input Image")
plt.axis("off")
plt.show()

**Loading the YOLOv5 model**

In [ ]:
yolo_model = torch.hub.load('ultralytics/yolov5', 'yolov5s', pretrained=True)
yolo_model.conf = 0.25
yolo_model.to(device)
yolo_model.eval()

print("YOLOv5 model loaded successfully.")

**Loading the DeepLabV3+ model**

In [ ]:
deeplab_model = smp.DeepLabV3Plus(
    encoder_name="resnet50",
    encoder_weights="imagenet",
    in_channels=3,
    classes=NUM_CLASSES,
    activation=None
).to(device)

checkpoint_path = "/content/deeplabv3plus_cityscapes.pth"

if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=device)

    if isinstance(checkpoint, dict) and "state_dict" in checkpoint:
        state_dict = checkpoint["state_dict"]
    else:
        state_dict = checkpoint

    cleaned_state_dict = {}
    for k, v in state_dict.items():
        new_k = k.replace("module.", "")
        cleaned_state_dict[new_k] = v

    deeplab_model.load_state_dict(cleaned_state_dict, strict=False)
    print("Loaded checkpoint:", checkpoint_path)
else:
    print("WARNING: No checkpoint found.")
    print("The code will still run, but segmentation quality may be poor unless you upload your trained .pth file.")

deeplab_model.eval()

**DeepLab preprocessing helper**

In [ ]:
def preprocess_for_deeplab(image_np, input_size=512):
    image_resized = cv2.resize(image_np, (input_size, input_size))
    image_resized = image_resized.astype(np.float32) / 255.0

    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

    image_norm = (image_resized - mean) / std
    image_tensor = torch.tensor(image_norm).permute(2, 0, 1).unsqueeze(0).float().to(device)

    return image_tensor

**Running the YOLOv5 and measure time**

In [ ]:
# warm-up
_ = yolo_model(image)

start_yolo = time.time()
yolo_results = yolo_model(image)
end_yolo = time.time()

yolo_time = end_yolo - start_yolo
yolo_fps = 1.0 / yolo_time if yolo_time > 0 else 0

print(f"YOLOv5 inference time: {yolo_time*1000:.2f} ms")
print(f"YOLOv5 FPS: {yolo_fps:.2f}")

**Showing the YOLO result**

In [ ]:
yolo_render = yolo_results.render()[0]   # returns image with boxes
yolo_render_rgb = cv2.cvtColor(yolo_render, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(8,5))
plt.imshow(yolo_render_rgb)
plt.title("YOLOv5 Detection Output")
plt.axis("off")
plt.show()

**Printing the YOLOv5 detections**

In [ ]:
detections_df = yolo_results.pandas().xyxy[0]
print(detections_df)
print("\nNumber of detected objects:", len(detections_df))

**Running the DeepLabV3+ and measuring time**

In [ ]:
image_tensor = preprocess_for_deeplab(image_np, input_size=512)

# warm-up
with torch.no_grad():
    _ = deeplab_model(image_tensor)

start_seg = time.time()
with torch.no_grad():
    logits = deeplab_model(image_tensor)
end_seg = time.time()

seg_time = end_seg - start_seg
seg_fps = 1.0 / seg_time if seg_time > 0 else 0

print(f"DeepLabV3+ inference time: {seg_time*1000:.2f} ms")
print(f"DeepLabV3+ FPS: {seg_fps:.2f}")

**Converting the segmentation output back to original size**

In [ ]:
upsampled_logits = F.interpolate(
    logits,
    size=(orig_h, orig_w),
    mode="bilinear",
    align_corners=False
)

pred_mask = upsampled_logits.argmax(dim=1)[0].cpu().numpy().astype(np.uint8)

print("Prediction mask shape:", pred_mask.shape)
print("Predicted class IDs:", np.unique(pred_mask))

**Creating the segmentation overlay**

In [ ]:
color_mask = CITYSCAPES_COLORS[pred_mask]
seg_overlay = (0.6 * image_np + 0.4 * color_mask).astype(np.uint8)

plt.figure(figsize=(14,5))

plt.subplot(1,2,1)
plt.imshow(color_mask)
plt.title("DeepLabV3+ Predicted Mask")
plt.axis("off")

plt.subplot(1,2,2)
plt.imshow(seg_overlay)
plt.title("DeepLabV3+ Overlay")
plt.axis("off")

plt.tight_layout()
plt.show()

**Printing the predicted segmentation classes**

In [ ]:
present_ids = np.unique(pred_mask)

print("Classes predicted in the image:")
for class_id in present_ids:
    if 0 <= class_id < len(CITYSCAPES_CLASSES):
        print(f"{class_id:2d}: {CITYSCAPES_CLASSES[class_id]}")

**Creating the final fused output**

In [ ]:
fused_image = seg_overlay.copy()

for _, row in detections_df.iterrows():
    x1, y1, x2, y2 = int(row["xmin"]), int(row["ymin"]), int(row["xmax"]), int(row["ymax"])
    label = f'{row["name"]} {row["confidence"]:.2f}'

    cv2.rectangle(fused_image, (x1, y1), (x2, y2), (0, 255, 0), 2)
    cv2.putText(
        fused_image,
        label,
        (x1, max(y1 - 10, 20)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.5,
        (255, 255, 255),
        2,
        cv2.LINE_AA
    )

plt.figure(figsize=(10,6))
plt.imshow(fused_image)
plt.title("Combined Pipeline Output (YOLOv5 + DeepLabV3+)")
plt.axis("off")
plt.show()

**Measuring the total pipeline time**

In [ ]:
total_pipeline_time = yolo_time + seg_time
combined_fps = 1.0 / total_pipeline_time if total_pipeline_time > 0 else 0

print(f"YOLOv5 time       : {yolo_time*1000:.2f} ms")
print(f"DeepLabV3+ time   : {seg_time*1000:.2f} ms")
print(f"Total pipeline    : {total_pipeline_time*1000:.2f} ms")
print(f"Combined FPS      : {combined_fps:.2f}")

**Saving the final outputs**

In [ ]:
Image.fromarray(yolo_render_rgb).save("/content/yolo_output.png")
Image.fromarray(color_mask).save("/content/deeplab_mask.png")
Image.fromarray(seg_overlay).save("/content/deeplab_overlay.png")
Image.fromarray(fused_image).save("/content/combined_pipeline_output.png")

print("Saved:")
print("/content/yolo_output.png")
print("/content/deeplab_mask.png")
print("/content/deeplab_overlay.png")
print("/content/combined_pipeline_output.png")

**Displaying the final evaluation summary table**

In [ ]:
print("="*50)
print("COMBINED PIPELINE EVALUATION SUMMARY")
print("="*50)
print(f"Detected objects           : {len(detections_df)}")
print(f"YOLOv5 inference time      : {yolo_time*1000:.2f} ms")
print(f"DeepLabV3+ inference time  : {seg_time*1000:.2f} ms")
print(f"Total pipeline time        : {total_pipeline_time*1000:.2f} ms")
print(f"Combined pipeline FPS      : {combined_fps:.2f}")
print(f"Segmentation classes found : {[CITYSCAPES_CLASSES[i] for i in present_ids if i < len(CITYSCAPES_CLASSES)]}")
print("="*50)

**Discovering the ground-truth segmentation mask**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
gt_mask_path = "/content/mask.png"

def compute_iou_per_class(pred, target, num_classes):
    ious = []
    for cls in range(num_classes):
        pred_inds = (pred == cls)
        target_inds = (target == cls)

        intersection = np.logical_and(pred_inds, target_inds).sum()
        union = np.logical_or(pred_inds, target_inds).sum()

        if union == 0:
            ious.append(np.nan)
        else:
            ious.append(intersection / union)
    return ious

if os.path.exists(gt_mask_path):
    gt_mask = np.array(Image.open(gt_mask_path))

    if gt_mask.ndim == 3:
        gt_h, gt_w, _ = gt_mask.shape
        class_mask = np.zeros((gt_h, gt_w), dtype=np.int32)

        for class_id, color in enumerate(CITYSCAPES_COLORS):
            same = np.all(gt_mask == color, axis=-1)
            class_mask[same] = class_id

        gt_mask = class_mask

    if gt_mask.shape != pred_mask.shape:
        gt_mask = cv2.resize(
            gt_mask.astype(np.int32),
            (pred_mask.shape[1], pred_mask.shape[0]),
            interpolation=cv2.INTER_NEAREST
        )

    ious = compute_iou_per_class(pred_mask.astype(np.int32), gt_mask.astype(np.int32), NUM_CLASSES)
    miou = np.nanmean(ious)

    print("\nPer-class IoU:")
    for i, iou in enumerate(ious):
        class_name = CITYSCAPES_CLASSES[i]
        if np.isnan(iou):
            print(f"{i:2d} ({class_name:15s}): N/A")
        else:
            print(f"{i:2d} ({class_name:15s}): {iou:.4f}")

    print(f"\nMean IoU (mIoU): {miou:.4f}")
else:
    print("No ground-truth mask found, so mIoU was not computed.")